# Setorização 5.6 – Notebook para Google Colab

### Novidades 5.6:
- **Raio por maioria de lojas**: o porte da rota (capital ou interior) é reavaliado a cada loja adicionada. Se a maioria das lojas for de cidades de interior, a rota passa a usar raios de interior — e vice-versa

### Acumulado das versões anteriores:
- 3 raios automáticos em sequência | Penalidade por dispersão geográfica
- Raios distintos para capitais e interior | Otimização de territórios (Bloco 4B)
- Nome da rota no padrão `SP | CAMPINAS - 001 (ROTA 174)` | Coluna CIDADE_CONTRATACAO
- Círculo de raio no mapa ao clicar na rota | Mapa HTML interativo | Excel com métricas


In [ ]:
# Bloco 1 — Parâmetros
# ► Este é o único bloco que você precisa editar antes de rodar.

!pip install -q haversine folium

import pandas as pd
import numpy as np
import folium
import math
from branca.element import Element
from haversine import haversine, Unit
from collections import defaultdict

# ============================================================
#   RAIOS POR PORTE DE CIDADE
#
#   CAPITAIS_E_METROPOLES
#   → Cidades com alta densidade urbana e bom transporte público.
#     Use raios menores para evitar rotas invasoras de território.
#     Adicione ou remova cidades conforme sua operação.
#
#   RAIOS_CAPITAL  = (Raio1, Raio2, Raio3) em km para capitais
#   RAIOS_INTERIOR = (Raio1, Raio2, Raio3) em km para demais cidades
#
#   Raio 1 → primeira tentativa (lojas mais próximas)
#   Raio 2 → residual do Raio 1 (amplia o alcance)
#   Raio 3 → residual do Raio 2 (aloca tudo que restar)
# ============================================================

CAPITAIS_E_METROPOLES = {
    # Capitais estaduais
    'SAO PAULO', 'RIO DE JANEIRO', 'BELO HORIZONTE', 'SALVADOR',
    'FORTALEZA', 'CURITIBA', 'MANAUS', 'RECIFE', 'PORTO ALEGRE',
    'BELEM', 'GOIANIA', 'FLORIANOPOLIS', 'MACEIO', 'NATAL',
    'TERESINA', 'CAMPO GRANDE', 'JOAO PESSOA', 'ARACAJU',
    'CUIABA', 'MACAPA', 'PORTO VELHO', 'RIO BRANCO', 'BOA VISTA',
    'PALMAS',
    # Grandes metrópoles / cidades com metrô ou BRT relevante
    'GUARULHOS', 'CAMPINAS', 'SAO BERNARDO DO CAMPO', 'SANTO ANDRE',
    'OSASCO', 'RIBEIRAO PRETO', 'SOROCABA', 'SAO JOSE DOS CAMPOS',
    'UBERLANDIA', 'CONTAGEM', 'JUIZ DE FORA', 'FEIRA DE SANTANA',
    'JOINVILLE', 'LONDRINA', 'APARECIDA DE GOIANIA', 'ANANINDEUA',
    'DUQUE DE CAXIAS', 'NOVA IGUACU', 'SAO LUIS', 'MOGI DAS CRUZES',
}

#   ┌─────────────────────────────────────────────────────────┐
#   │  Edite os valores de KM abaixo conforme necessário      │
#   └─────────────────────────────────────────────────────────┘

RAIOS_CAPITAL  = (4.0,  8.0, 12.0)   # km — capitais e metrópoles
RAIOS_INTERIOR = (5.0, 10.0, 20.0)   # km — cidades do interior

# ============================================================
#   CAPACIDADE DO PROMOTOR
#   CAPACIDADE_BASE.......: horas semanais contratadas
#   LIMITE_LOJAS_51H......: até quantas lojas o roteiro pode ter 51h
#   BUFFER_KM.............: km extras além do raio (loja próxima à mais extrema)
# ============================================================
CAPACIDADE_BASE  = 44.0
LIMITE_LOJAS_51H = 3
BUFFER_KM        = 2.0

# ============================================================
#   DESLOCAMENTO ESTIMADO
#   KM_POR_PERNA.........: distância média entre lojas consecutivas
#   VELOCIDADE_KM_H......: velocidade média de deslocamento
#   TEMPO_FIXO_PERNA_H...: tempo fixo de parada por perna (em horas)
# ============================================================
KM_POR_PERNA       = 1.8
VELOCIDADE_KM_H    = 18.0
TEMPO_FIXO_PERNA_H = 10.0 / 60.0

# ============================================================
#   CONTROLE DE DISPERSÃO GEOGRÁFICA
#   MAX_DISPERSAO_KM: desvio padrão máximo das distâncias ao centróide
#   Menor = rotas mais compactas  |  float('inf') = desativado
# ============================================================
MAX_DISPERSAO_KM = 6.0

# ============================================================
#   OTIMIZAÇÃO DE TERRITÓRIOS (Bloco 4B)
#   MAX_ITER_TERRITORIO: rodadas de troca entre rotas vizinhas
#   Recomendado: 3 a 10  |  0 = desativado
# ============================================================
MAX_ITER_TERRITORIO = 5

# ============================================================
#   FAIXAS BOAS — rotas aceitas (não entram no residual)
#   Opções: '44h ou mais' | '40 a 43h' | '35 a 39h' | '30 a 34h' | '22 a 29h' | 'Abaixo de 22h'
# ============================================================
FAIXAS_BOAS = {'44h ou mais', '40 a 43h'}

# ============================================================

def porte_cidade(nome_cidade):
    """Retorna 'capital' ou 'interior' para a cidade informada."""
    nome = str(nome_cidade).strip().upper()
    # normaliza acentos básicos para comparação
    for a, b in [('Ã','A'),('Á','A'),('À','A'),('Â','A'),('É','E'),('Ê','E'),
                 ('Í','I'),('Ó','O'),('Ô','O'),('Õ','O'),('Ú','U'),('Ç','C')]:
        nome = nome.replace(a, b)
    return 'capital' if nome in CAPITAIS_E_METROPOLES else 'interior'

def raios_para_cidade(nome_cidade):
    """Retorna a tupla (raio1, raio2, raio3) conforme o porte da cidade."""
    return RAIOS_CAPITAL if porte_cidade(nome_cidade) == 'capital' else RAIOS_INTERIOR

def tempo_deslocamento(n_lojas):
    pernas = max(0, int(n_lojas) - 1)
    return pernas * (KM_POR_PERNA / VELOCIDADE_KM_H + TEMPO_FIXO_PERNA_H)

def capacidade_efetiva(n_lojas):
    return CAPACIDADE_BASE

def classificar_faixa_horas(horas):
    if horas >= 44:       return '44h ou mais'
    if 40 <= horas < 44:  return '40 a 43h'
    if 35 <= horas < 40:  return '35 a 39h'
    if 30 <= horas < 35:  return '30 a 34h'
    if 22 <= horas < 30:  return '22 a 29h'
    return 'Abaixo de 22h'

def distancia_km(p1, p2):
    lat1, lon1 = p1; lat2, lon2 = p2
    for v in [lat1, lon1, lat2, lon2]:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return float('inf')
    R = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def distancia_km_rapida(p1, p2):
    lat1, lon1 = p1; lat2, lon2 = p2
    for v in [lat1, lon1, lat2, lon2]:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return float('inf')
    R = 6371.0088
    x = math.radians(lon2 - lon1) * math.cos(math.radians((lat1+lat2)/2))
    y = math.radians(lat2 - lat1)
    return R * math.sqrt(x*x + y*y)

def encontrar_coluna(df, candidatos, obrigatoria=True):
    cols_lower = {c.lower(): c for c in df.columns}
    for nome in candidatos:
        if nome.lower() in cols_lower:
            return cols_lower[nome.lower()]
    if obrigatoria:
        raise KeyError(f'Coluna obrigatória não encontrada: {candidatos}')
    return None

print('Bloco 1 carregado.')
print(f'Raios CAPITAL:  R1={RAIOS_CAPITAL[0]} km | R2={RAIOS_CAPITAL[1]} km | R3={RAIOS_CAPITAL[2]} km')
print(f'Raios INTERIOR: R1={RAIOS_INTERIOR[0]} km | R2={RAIOS_INTERIOR[1]} km | R3={RAIOS_INTERIOR[2]} km')
print(f'Dispersão máxima: {MAX_DISPERSAO_KM} km | Iterações de território: {MAX_ITER_TERRITORIO}')


In [ ]:
# Bloco 2 — Upload e validação da base

from google.colab import files

MAPA_COLUNAS = {
    'CHAVE ESTUDO':  ['CHAVE ESTUDO', 'CHAVE_ESTUDO', 'CHAVE'],
    'CANAL':         ['CANAL', 'CANAL PDV', 'CANAL_PDV', 'CANAL RESUMIDO', 'CANAL_RESUMIDO'],
    'BANDEIRA':      ['BANDEIRA'],
    'CIDADE':        ['CIDADE'],
    'UF':            ['UF', 'ESTADO'],
    'LATITUDE':      ['LATITUDE', 'LAT'],
    'LONGITUDE':     ['LONGITUDE', 'LON', 'LONG'],
    'CAT':           ['CAT'],
    'FREQUÊNCIA':    ['FREQUÊNCIA', 'FREQUENCIA', 'FREQ'],
    'PERMANENCIA':   ['PERMANENCIA', 'PERMANÊNCIA'],
    'TEMPO_SEMANAL': ['TEMPO_SEMANAL', 'TEMPO SEMANAL', 'TEMPO_SEMANAL_H', 'TEMPO SEMANAL H'],
}

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
print('Arquivo recebido:', nome_arquivo)

if nome_arquivo.lower().endswith(('.xls', '.xlsx')):
    df_raw = pd.read_excel(nome_arquivo)
elif nome_arquivo.lower().endswith('.csv'):
    df_raw = pd.read_csv(nome_arquivo, sep=';', decimal=',')
else:
    raise ValueError('Formato não suportado. Use .xlsx, .xls ou .csv')

df_raw.columns = [c.strip() for c in df_raw.columns]

rename_map = {}
nao_encontradas = []
for col_padrao, candidatos in MAPA_COLUNAS.items():
    encontrada = encontrar_coluna(df_raw, candidatos, obrigatoria=False)
    if encontrada:
        if encontrada != col_padrao:
            rename_map[encontrada] = col_padrao
    else:
        nao_encontradas.append(col_padrao)

if nao_encontradas:
    raise ValueError(
        f'ERRO: Colunas não encontradas: {nao_encontradas}\n'
        f'Colunas no arquivo: {list(df_raw.columns)}'
    )

df_raw = df_raw.rename(columns=rename_map)
df_raw = df_raw[list(MAPA_COLUNAS.keys())]

print(f'Base lida. Linhas: {len(df_raw)}')
df_raw.head()


In [ ]:
# Bloco 3 — Preparação

df = df_raw.copy()

col_lat   = 'LATITUDE'
col_lon   = 'LONGITUDE'
col_tempo = 'TEMPO_SEMANAL'
col_uf    = 'UF'
col_cidade = 'CIDADE'
col_nome  = 'BANDEIRA'

df[col_lat]   = pd.to_numeric(df[col_lat],   errors='coerce')
df[col_lon]   = pd.to_numeric(df[col_lon],   errors='coerce')
df[col_tempo] = pd.to_numeric(df[col_tempo], errors='coerce')

n_antes = len(df)
df = df.dropna(subset=[col_lat, col_lon, col_tempo]).copy().reset_index(drop=True)
if len(df) < n_antes:
    print(f'AVISO: {n_antes - len(df)} linha(s) removida(s) por dados inválidos.')

df['ROTA']          = None
df['ALOCADA']       = False
df['RAIO_FORMACAO'] = None

print(f'Linhas válidas: {len(df)}')
print(f'UFs: {sorted(df[col_uf].unique())}')


In [ ]:
# Bloco 4 — Algoritmo (3 raios automáticos)

rotas_info       = []
rota_seq_global  = 1          # contador global de rotas
seq_por_cidade   = defaultdict(int)  # contador por 'UF|CIDADE'

def cidade_dominante(rota_idxs):
    horas_por_cidade = defaultdict(float)
    for i in rota_idxs:
        cidade = str(df.at[i, col_cidade]).strip().upper()
        horas_por_cidade[cidade] += float(df.at[i, col_tempo])
    return max(horas_por_cidade, key=horas_por_cidade.get)

def montar_nome_rota(uf, cidade, seq_cidade, seq_global):
    return f'{uf} | {cidade} - {seq_cidade:03d} (ROTA {seq_global})'

def porte_majoritario(rota_idxs):
    """Retorna 'capital' se a maioria das lojas for de cidades capital,
    caso contrário retorna 'interior'."""
    contagem = {'capital': 0, 'interior': 0}
    for i in rota_idxs:
        cidade = str(df.at[i, col_cidade]).strip().upper()
        contagem[porte_cidade(cidade)] += 1
    return 'capital' if contagem['capital'] >= contagem['interior'] else 'interior'

def raio_atual(rota_idxs, num_raio):
    """Retorna o raio em km baseado no porte majoritário da rota no momento."""
    porte = porte_majoritario(rota_idxs)
    return RAIOS_CAPITAL[num_raio - 1] if porte == 'capital' else RAIOS_INTERIOR[num_raio - 1]

def rodar_raio(df, num_raio, nome_raio, rota_seq_global, aceitar_todas=False):
    rotas_novas = []
    faixas_ok   = set(['44h ou mais','40 a 43h','35 a 39h','30 a 34h','22 a 29h','Abaixo de 22h']) if aceitar_todas else FAIXAS_BOAS

    group_cols = [col_uf]
    for c in df.columns:
        if any(k in c.lower() for k in ['hidr', 'ponte', 'barre']):
            group_cols = [col_uf, c]
            break

    for group_vals, idxs in df.groupby(group_cols).groups.items():
        nao_alocados = set(i for i in idxs if not df.at[i, 'ALOCADA'])
        if not nao_alocados:
            continue

        while nao_alocados:
            seed_idx  = max(nao_alocados, key=lambda i: float(df.at[i, col_tempo]))
            uf_val    = str(df.at[seed_idx, col_uf]).strip().upper()
            cidade_seed = str(df.at[seed_idx, col_cidade]).strip().upper()

            rota_idxs = [seed_idx]
            nao_alocados.discard(seed_idx)

            melhorou = True
            while melhorou and nao_alocados:
                melhorou = False
                lat_c = float(df.loc[rota_idxs, col_lat].mean())
                lon_c = float(df.loc[rota_idxs, col_lon].mean())
                soma_tempos = float(df.loc[rota_idxs, col_tempo].sum())
                n_lojas     = len(rota_idxs)

                candidatos_ord = sorted(
                    list(nao_alocados),
                    key=lambda i: distancia_km_rapida(
                        (lat_c, lon_c),
                        (float(df.at[i, col_lat]), float(df.at[i, col_lon]))
                    )
                )

                dists   = [distancia_km_rapida((lat_c, lon_c),
                           (float(df.at[i, col_lat]), float(df.at[i, col_lon])))
                           for i in rota_idxs]
                idx_ext = rota_idxs[int(np.argmax(dists))]
                lat_ext = float(df.at[idx_ext, col_lat])
                lon_ext = float(df.at[idx_ext, col_lon])

                raio_km = raio_atual(rota_idxs, num_raio)

                for idx_cand in candidatos_ord:
                    lat_c2 = float(df.at[idx_cand, col_lat])
                    lon_c2 = float(df.at[idx_cand, col_lon])
                    if distancia_km((lat_c, lon_c), (lat_c2, lon_c2)) > raio_km and \
                       distancia_km((lat_ext, lon_ext), (lat_c2, lon_c2)) > BUFFER_KM:
                        continue

                    t_loja     = float(df.at[idx_cand, col_tempo])
                    novo_n     = n_lojas + 1
                    novo_total = soma_tempos + t_loja + tempo_deslocamento(novo_n)
                    limite     = 51 if novo_n <= LIMITE_LOJAS_51H else capacidade_efetiva(novo_n)

                    if novo_total <= limite:
                        # verifica dispersão: calcula desvio padrão das distâncias
                        # ao centróide após incluir a candidata
                        candidatos_com_nova = rota_idxs + [idx_cand]
                        lat_novo_c = float(df.loc[candidatos_com_nova, col_lat].mean())
                        lon_novo_c = float(df.loc[candidatos_com_nova, col_lon].mean())
                        dists_internas = [
                            distancia_km_rapida(
                                (lat_novo_c, lon_novo_c),
                                (float(df.at[i, col_lat]), float(df.at[i, col_lon]))
                            ) for i in candidatos_com_nova
                        ]
                        dispersao = float(np.std(dists_internas)) if len(dists_internas) > 1 else 0.0
                        if dispersao > MAX_DISPERSAO_KM:
                            continue
                        rota_idxs.append(idx_cand)
                        nao_alocados.discard(idx_cand)
                        soma_tempos += t_loja
                        n_lojas      = novo_n
                        melhorou     = True
                        break

            n_final    = len(rota_idxs)
            soma_final = float(df.loc[rota_idxs, col_tempo].sum())
            desloc_f   = tempo_deslocamento(n_final)
            total_f    = soma_final + desloc_f
            faixa_f    = classificar_faixa_horas(total_f)

            if faixa_f in faixas_ok:
                cidade_dom = cidade_dominante(rota_idxs)
                chave_cidade = f'{uf_val}|{cidade_dom}'
                seq_por_cidade[chave_cidade] += 1
                seq_cid = seq_por_cidade[chave_cidade]
                rota_nome = montar_nome_rota(uf_val, cidade_dom, seq_cid, rota_seq_global)
                rota_seq_global += 1

                for i in rota_idxs:
                    df.at[i, 'ALOCADA']       = True
                    df.at[i, 'ROTA']          = rota_nome
                    df.at[i, 'RAIO_FORMACAO'] = nome_raio

                # calcula raio real da rota (distância máxima do centróide)
                lat_fin = float(df.loc[rota_idxs, col_lat].mean())
                lon_fin = float(df.loc[rota_idxs, col_lon].mean())
                raio_real_km = max(
                    distancia_km((lat_fin, lon_fin),
                                 (float(df.at[i, col_lat]), float(df.at[i, col_lon])))
                    for i in rota_idxs
                )
                rotas_novas.append({
                    'nome':         rota_nome,
                    'idxs':         rota_idxs,
                    'lat':          lat_fin,
                    'lon':          lon_fin,
                    't_loja_h':     soma_final,
                    't_desloc_h':   desloc_f,
                    't_total_h':    total_f,
                    'qtd_lojas':    n_final,
                    'faixa':        faixa_f,
                    'raio':         nome_raio,
                    'porte':        porte_majoritario(rota_idxs),
                    'raio_real_km': round(raio_real_km, 2),
                    'uf':           uf_val,
                    'cidade_dom':   cidade_dom,
                })
            else:
                for i in rota_idxs:
                    df.at[i, 'ALOCADA']       = False
                    df.at[i, 'ROTA']          = None
                    df.at[i, 'RAIO_FORMACAO'] = None

    return rotas_novas, rota_seq_global


print(f'=== Raio 1 — Capital: {RAIOS_CAPITAL[0]} km | Interior: {RAIOS_INTERIOR[0]} km ===')
novas, rota_seq_global = rodar_raio(df, 1, 'Raio 1', rota_seq_global)
rotas_info.extend(novas)
n_r1 = df['ALOCADA'].sum()
print(f'Concluído: {n_r1} lojas alocadas | {(~df["ALOCADA"]).sum()} no residual')

print(f'\n=== Raio 2 — Capital: {RAIOS_CAPITAL[1]} km | Interior: {RAIOS_INTERIOR[1]} km (residual Raio 1) ===')
novas, rota_seq_global = rodar_raio(df, 2, 'Raio 2', rota_seq_global)
rotas_info.extend(novas)
n_r2 = df['ALOCADA'].sum() - n_r1
print(f'Concluído: +{n_r2} lojas alocadas | {(~df["ALOCADA"]).sum()} no residual')

print(f'\n=== Raio 3 — Capital: {RAIOS_CAPITAL[2]} km | Interior: {RAIOS_INTERIOR[2]} km (residual Raio 2 — aloca tudo) ===')
novas, rota_seq_global = rodar_raio(df, 3, 'Raio 3', rota_seq_global, aceitar_todas=True)
rotas_info.extend(novas)
print(f'Concluído: {df["ALOCADA"].sum()} / {len(df)} lojas alocadas')

print(f'\n=== Setorização concluída | Total de rotas: {len(rotas_info)} ===')
for rn in ['Raio 1', 'Raio 2', 'Raio 3']:
    rr = [r for r in rotas_info if r['raio'] == rn]
    boas = sum(1 for r in rr if r['faixa'] in FAIXAS_BOAS)
    print(f'  {rn}: {len(rr)} rotas ({boas} boas | {len(rr)-boas} abaixo de 40h)')


In [ ]:
# Bloco 4B — Otimização de territórios (reduz invasões entre rotas)

def otimizar_territorios(df, rotas_info, max_iter=5):
    if max_iter == 0:
        print('Otimização de territórios desativada (MAX_ITER_TERRITORIO = 0).')
        return rotas_info

    # Índice rápido: loja → índice da rota em rotas_info
    loja_para_rota = {}
    for r_idx, r in enumerate(rotas_info):
        for i in r['idxs']:
            loja_para_rota[i] = r_idx

    trocas_total = 0

    for iteracao in range(max_iter):
        trocas_iter = 0

        # Recalcula centróides antes de cada iteração
        centroide = {}
        for r_idx, r in enumerate(rotas_info):
            lats = [float(df.at[i, col_lat]) for i in r['idxs']]
            lons = [float(df.at[i, col_lon]) for i in r['idxs']]
            centroide[r_idx] = (sum(lats)/len(lats), sum(lons)/len(lons))

        # Para cada loja, verifica se ela está mais perto do centróide de outra rota
        for loja_idx, r_orig_idx in list(loja_para_rota.items()):
            r_orig = rotas_info[r_orig_idx]

            # Rota com apenas 1 loja não pode ceder
            if len(r_orig['idxs']) <= 1:
                continue

            lat_loja = float(df.at[loja_idx, col_lat])
            lon_loja = float(df.at[loja_idx, col_lon])
            t_loja   = float(df.at[loja_idx, col_tempo])

            dist_orig = distancia_km_rapida(centroide[r_orig_idx], (lat_loja, lon_loja))

            # Procura rota vizinha cujo centróide seja mais próximo desta loja
            melhor_destino = None
            melhor_dist    = dist_orig

            for r_dest_idx, r_dest in enumerate(rotas_info):
                if r_dest_idx == r_orig_idx:
                    continue
                # Só considera rotas da mesma UF
                if r_dest.get('uf') != r_orig.get('uf'):
                    continue

                dist_dest = distancia_km_rapida(centroide[r_dest_idx], (lat_loja, lon_loja))
                if dist_dest >= melhor_dist:
                    continue

                # Verifica se a rota destino ainda cabe a loja em horas
                n_dest     = len(r_dest['idxs'])
                novo_total_dest = r_dest['t_loja_h'] + t_loja + tempo_deslocamento(n_dest + 1)
                limite_dest = 51 if (n_dest + 1) <= LIMITE_LOJAS_51H else capacidade_efetiva(n_dest + 1)
                if novo_total_dest > limite_dest:
                    continue

                # Verifica se a rota origem não fica inviável sem esta loja
                n_orig      = len(r_orig['idxs'])
                novo_total_orig = r_orig['t_loja_h'] - t_loja + tempo_deslocamento(n_orig - 1)
                # Aceita mesmo que a origem fique abaixo das faixas boas
                # (preferimos territórios limpos a forçar horas)
                if novo_total_orig < 0:
                    continue

                # Verifica dispersão na rota destino após inclusão
                idxs_dest_novo = r_dest['idxs'] + [loja_idx]
                lat_novo_c = float(df.loc[idxs_dest_novo, col_lat].mean())
                lon_novo_c = float(df.loc[idxs_dest_novo, col_lon].mean())
                dists_d = [distancia_km_rapida((lat_novo_c, lon_novo_c),
                           (float(df.at[i, col_lat]), float(df.at[i, col_lon])))
                           for i in idxs_dest_novo]
                if float(np.std(dists_d)) > MAX_DISPERSAO_KM:
                    continue

                melhor_dist    = dist_dest
                melhor_destino = r_dest_idx

            if melhor_destino is None:
                continue

            # Executa a troca
            r_orig = rotas_info[r_orig_idx]
            r_dest = rotas_info[melhor_destino]

            r_orig['idxs'].remove(loja_idx)
            r_dest['idxs'].append(loja_idx)
            loja_para_rota[loja_idx] = melhor_destino

            # Atualiza métricas das duas rotas
            for r in [r_orig, r_dest]:
                n = len(r['idxs'])
                soma = float(df.loc[r['idxs'], col_tempo].sum())
                desloc = tempo_deslocamento(n)
                total  = soma + desloc
                lat_c  = float(df.loc[r['idxs'], col_lat].mean())
                lon_c  = float(df.loc[r['idxs'], col_lon].mean())
                raio_real = max(
                    distancia_km((lat_c, lon_c),
                                 (float(df.at[i, col_lat]), float(df.at[i, col_lon])))
                    for i in r['idxs']
                )
                r.update({
                    't_loja_h':   soma,
                    't_desloc_h': desloc,
                    't_total_h':  total,
                    'qtd_lojas':  n,
                    'faixa':      classificar_faixa_horas(total),
                    'lat':        lat_c,
                    'lon':        lon_c,
                    'raio_real_km': round(raio_real, 2),
                })

            # Atualiza df
            df.at[loja_idx, 'ROTA'] = r_dest['nome']
            trocas_iter   += 1
            trocas_total  += 1

        print(f'  Iteração {iteracao+1}: {trocas_iter} troca(s)')
        if trocas_iter == 0:
            print('  Convergiu — nenhuma melhoria adicional possível.')
            break

    print(f'Total de trocas realizadas: {trocas_total}')
    return rotas_info


print(f'=== Bloco 4B — Otimização de territórios ({MAX_ITER_TERRITORIO} iterações máx.) ===')
rotas_info = otimizar_territorios(df, rotas_info, max_iter=MAX_ITER_TERRITORIO)
print('Otimização concluída.')


In [ ]:
# Bloco 5 — Métricas

resultado = df.copy().reset_index(drop=True)
resultado[col_tempo] = pd.to_numeric(resultado[col_tempo], errors='coerce').fillna(0)
resultado['ROTA']    = resultado['ROTA'].astype('string')

resultado['T_LOJA_H']  = resultado.groupby('ROTA')[col_tempo].transform('sum')
resultado['QTD_LOJAS'] = resultado.groupby('ROTA')['ROTA'].transform('count')
resultado['T_LOJA_H']  = resultado['T_LOJA_H'].fillna(0)
resultado['QTD_LOJAS'] = resultado['QTD_LOJAS'].fillna(0).astype(int)
resultado['T_DESLO_H'] = resultado['QTD_LOJAS'].apply(tempo_deslocamento)
resultado['T_TOTAL_H'] = resultado['T_LOJA_H'] + resultado['T_DESLO_H']
resultado['CAP_EFETIVA'] = resultado['QTD_LOJAS'].apply(capacidade_efetiva)
resultado['OCUPACAO']  = resultado['T_TOTAL_H'] / resultado['CAP_EFETIVA']
resultado['FAIXA_HORAS'] = resultado['T_TOTAL_H'].apply(classificar_faixa_horas)

info_por_rota = {r['nome']: r for r in rotas_info}
resultado['CIDADE_CONTRATACAO'] = resultado['ROTA'].apply(
    lambda nome: f"{info_por_rota[nome]['uf']}-{info_por_rota[nome]['cidade_dom']}"
    if nome in info_por_rota else ''
)

print('Métricas calculadas. Preview:')
resultado[['ROTA', 'CIDADE_CONTRATACAO', 'RAIO_FORMACAO', 'QTD_LOJAS', 'T_TOTAL_H', 'FAIXA_HORAS']].drop_duplicates('ROTA').head(10)


In [ ]:
# Bloco 6 — Exportar Excel

colunas_saida = [
    'CHAVE ESTUDO', 'ROTA', 'CIDADE_CONTRATACAO', 'RAIO_FORMACAO', 'FAIXA_HORAS',
    'CANAL', 'BANDEIRA', 'CIDADE', 'UF',
    'CAT', 'FREQUÊNCIA', 'PERMANENCIA', 'TEMPO_SEMANAL',
    'LATITUDE', 'LONGITUDE',
    'QTD_LOJAS', 'T_LOJA_H', 'T_DESLO_H', 'T_TOTAL_H', 'CAP_EFETIVA', 'OCUPACAO'
]
colunas_saida = [c for c in colunas_saida if c in resultado.columns]

nome_saida = 'setorizacao_5_1_resultado.xlsx'
resultado[colunas_saida].to_excel(nome_saida, index=False)
print('Arquivo salvo:', nome_saida)

from google.colab import files
files.download(nome_saida)


In [ ]:
# Bloco 7 — Mapa HTML

import json
import folium
from branca.element import Element

PALETA_CORES = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
    '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
    '#bcbd22', '#17becf'
]

radius = 9.5
weight = 1
m = folium.Map(
    location=[resultado[col_lat].mean(), resultado[col_lon].mean()],
    zoom_start=7, tiles='CartoDB positron'
)

labels_ordenados = ['44h ou mais', '40 a 43h', '35 a 39h', '30 a 34h', '22 a 29h', 'Abaixo de 22h']

rota_meta = {r['nome']: r for r in rotas_info}
rota_layers = []

for idx_rota, info in enumerate(rotas_info):
    rota_nome  = info['nome']
    cor        = PALETA_CORES[idx_rota % len(PALETA_CORES)]
    meta       = rota_meta.get(rota_nome, {})
    faixa_txt  = meta.get('faixa', '')
    t_loja_h   = float(meta.get('t_loja_h', 0))
    t_desloc_h = float(meta.get('t_desloc_h', 0))
    qtd_lojas  = int(meta.get('qtd_lojas', 0))
    raio_txt   = meta.get('raio', '')
    cidade_dom = meta.get('cidade_dom', '')
    uf_val     = meta.get('uf', '')

    fg_rota = folium.FeatureGroup(name=rota_nome, show=True)
    lojas   = resultado[resultado['ROTA'] == rota_nome]
    if lojas.empty:
        continue

    lat_c = float(lojas[col_lat].mean())
    lon_c = float(lojas[col_lon].mean())
    dists_centro = lojas.apply(
        lambda r: distancia_km((lat_c, lon_c), (float(r[col_lat]), float(r[col_lon]))), axis=1)
    idx_hub = dists_centro.idxmin()
    hub_lat = float(lojas.loc[idx_hub, col_lat])
    hub_lon = float(lojas.loc[idx_hub, col_lon])

    for idx, row in lojas.iterrows():
        lat = float(row[col_lat])
        lon = float(row[col_lon])
        loja_nome = str(row[col_nome])

        tooltip_html = f"""
        <div class='spot-tooltip-card'>
          <div class='spot-tt-title'>{rota_nome}</div>
          <div class='spot-tt-city'>Contratação: {uf_val}-{cidade_dom}</div>
          <div class='spot-tt-store'>{loja_nome}</div>
          <div class='spot-tt-sep'></div>
          <div class='spot-tt-info'>
            <div>{faixa_txt} &bull; {raio_txt}</div>
            <div>{t_loja_h:.0f}h em loja / {t_desloc_h:.1f}h deslocamento</div>
            <div>{qtd_lojas} lojas no roteiro</div>
          </div>
        </div>"""

        folium.CircleMarker(
            location=(lat, lon), radius=radius, color=cor,
            fill=True, fill_color=cor, fill_opacity=0.8, weight=1,
            tooltip=folium.Tooltip(tooltip_html, sticky=False),
        ).add_to(fg_rota)

        if idx != idx_hub:
            folium.PolyLine(
                locations=[(hub_lat, hub_lon), (lat, lon)],
                color=cor, weight=weight, opacity=0.8, dash_array='5,5',
            ).add_to(fg_rota)

    fg_rota.add_to(m)
    rota_layers.append({
        'id':          idx_rota,
        'nome':        rota_nome,
        'faixa':       faixa_txt,
        'cor':         cor,
        'layer_var':   fg_rota.get_name(),
        'raio':        raio_txt,
        'lat':         float(info.get('lat', lat_c)),
        'lon':         float(info.get('lon', lon_c)),
        'raio_real_km': float(info.get('raio_real_km', 1.0)),
    })

SIDEBAR_TEMPLATE = '\n<style>\n.leaflet-tooltip{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;}\n.leaflet-tooltip:before{display:none!important;}\n.spot-tooltip-card{min-width:190px;max-width:270px;background:#fff;border-radius:12px;box-shadow:0 8px 22px rgba(0,0,0,.22);padding:10px 12px;font-family:Arial,sans-serif;line-height:1.25;}\n.spot-tt-title{font-weight:700;font-size:13px;margin-bottom:2px;}\n.spot-tt-city{font-size:11px;color:#666;margin-bottom:6px;}\n.spot-tt-store{font-weight:600;font-size:12px;margin-bottom:8px;}\n.spot-tt-sep{border-top:1px solid #e6e6e6;margin:8px 0;}\n.spot-tt-info{font-size:11px;color:#111;}\n.spot-tt-info div{margin-bottom:2px;}\n#spot-sidebar{position:absolute;top:80px;left:12px;z-index:1000;width:310px;max-height:90vh;overflow-y:auto;background:#fff;border-radius:14px;box-shadow:0 0 10px rgba(0,0,0,.18);padding:10px 10px 12px 10px;font-family:Arial,sans-serif;font-size:12px;}\n#spot-sidebar h3{margin:0 0 8px 0;font-size:13px;font-weight:600;}\n.spot-top-controls{display:flex;flex-direction:column;gap:6px;margin-bottom:8px;}\n#spot-search{width:100%;box-sizing:border-box;padding:6px 8px;border:1px solid #ddd;border-radius:10px;outline:none;font-size:11px;}\n.spot-global-buttons{display:flex;gap:6px;}\n.spot-global-buttons button{flex:1;padding:6px 8px;font-size:10px;border:1px solid #ccc;background:#fff;border-radius:10px;cursor:pointer;}\n.spot-faixa-block{margin-top:8px;padding:6px 8px;border-radius:10px;border:1px solid #eee;background:#fafafa;}\n.spot-faixa-title{display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;font-weight:600;font-size:11px;}\n.spot-faixa-buttons{display:flex;gap:3px;}\n.spot-faixa-buttons button{padding:2px 6px;font-size:10px;border:1px solid #ccc;background:#fff;border-radius:4px;cursor:pointer;}\n.spot-rota-item{display:flex;align-items:center;margin:2px 0;}\n.spot-rota-color{width:10px;height:10px;border-radius:50%;margin-right:6px;flex-shrink:0;}\n.spot-rota-checkbox{margin-right:4px;}\n.spot-rota-label{flex-grow:1;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;font-size:11px;cursor:pointer;}\n.spot-rota-label:hover{text-decoration:underline;}\n.spot-raio-badge{font-size:8px;background:#eee;border-radius:4px;padding:1px 4px;margin-left:4px;flex-shrink:0;color:#555;}\n</style>\n\n<div id="spot-sidebar">\n  <h3>Rotas por faixa de horas</h3>\n  <div class="spot-top-controls">\n    <input id="spot-search" type="text" placeholder="Buscar rota..." />\n    <div class="spot-global-buttons">\n      <button id="spot-btn-showall" type="button">Mostrar tudo</button>\n      <button id="spot-btn-hideall" type="button">Ocultar tudo</button>\n    </div>\n  </div>\n  <div id="spot-faixas-container"></div>\n</div>\n\n<script>\nconst SPOT_DATA = __SPOT_DATA__;\nlet SPOT_QUERY = "";\nlet activeCircle = null;\nlet activeIdx = null;\n\nfunction getMap() { return window[SPOT_DATA.map_var]; }\n\nfunction spotSetRouteVisible(idx, visible) {\n  const r = SPOT_DATA.rotas[idx], map = getMap(), layer = window[r.layer_var];\n  if (!map || !layer) return;\n  if (visible) { if (!map.hasLayer(layer)) map.addLayer(layer); }\n  else { if (map.hasLayer(layer)) map.removeLayer(layer); }\n}\n\nfunction spotFocusRoute(idx) {\n  const r = SPOT_DATA.rotas[idx];\n  const map = getMap();\n  if (!map) return;\n\n  if (activeCircle) { map.removeLayer(activeCircle); activeCircle = null; }\n\n  if (activeIdx === idx) {\n    activeIdx = null;\n    return;\n  }\n\n  activeIdx = idx;\n  map.setView([r.lat, r.lon], 12, {animate: true});\n\n  const raioNome = r.raio || \'\';\n  const raioKm   = r.raio_real_km || 1;\n  const cor      = r.cor || \'#333\';\n\n  activeCircle = L.circle([r.lat, r.lon], {\n    radius: raioKm * 1000,\n    color: cor,\n    weight: 2,\n    opacity: 0.9,\n    fillColor: cor,\n    fillOpacity: 0.08,\n  }).bindTooltip(\n    `<b>${r.nome}</b><br>${raioNome} &bull; raio real: ${raioKm} km`,\n    {sticky: false, direction: \'top\'}\n  ).addTo(map);\n}\n\nfunction spotApplyFilter(query) {\n  SPOT_QUERY = (query || "").toLowerCase().trim();\n  document.querySelectorAll(\'[data-spot-route-item="1"]\').forEach(el => {\n    const idx = parseInt(el.getAttribute("data-idx"), 10);\n    const match = (SPOT_QUERY === "" || (el.getAttribute("data-name") || "").indexOf(SPOT_QUERY) !== -1);\n    el.style.display = match ? "" : "none";\n    const cb = document.getElementById("spot-rota-cb-" + idx);\n    spotSetRouteVisible(idx, match && cb && cb.checked);\n  });\n}\n\nfunction spotShowAll() { SPOT_DATA.rotas.forEach((r,idx) => { const cb = document.getElementById("spot-rota-cb-"+idx); if(cb) cb.checked=true; }); spotApplyFilter(SPOT_QUERY); }\nfunction spotHideAll() { SPOT_DATA.rotas.forEach((r,idx) => { const cb = document.getElementById("spot-rota-cb-"+idx); if(cb) cb.checked=false; }); spotApplyFilter(SPOT_QUERY); }\n\nfunction spotInitSidebar() {\n  const container = document.getElementById("spot-faixas-container");\n  if (!container) return;\n  SPOT_DATA.faixas.forEach(faixa => {\n    const bloco = document.createElement("div"); bloco.className = "spot-faixa-block";\n    const titulo = document.createElement("div"); titulo.className = "spot-faixa-title";\n    const spanTit = document.createElement("span"); spanTit.textContent = faixa; titulo.appendChild(spanTit);\n    const btnOn = document.createElement("button"); btnOn.textContent = "Mostrar";\n    btnOn.onclick = () => { SPOT_DATA.rotas.forEach((r,idx) => { if(r.faixa===faixa){ const cb=document.getElementById("spot-rota-cb-"+idx); if(cb&&!cb.checked) cb.checked=true; } }); spotApplyFilter(SPOT_QUERY); };\n    const btnOff = document.createElement("button"); btnOff.textContent = "Ocultar";\n    btnOff.onclick = () => { SPOT_DATA.rotas.forEach((r,idx) => { if(r.faixa===faixa){ const cb=document.getElementById("spot-rota-cb-"+idx); if(cb&&cb.checked) cb.checked=false; } }); spotApplyFilter(SPOT_QUERY); };\n    const btnBox = document.createElement("div"); btnBox.className="spot-faixa-buttons"; btnBox.appendChild(btnOn); btnBox.appendChild(btnOff);\n    titulo.appendChild(btnBox); bloco.appendChild(titulo);\n    SPOT_DATA.rotas.forEach((r,idx) => {\n      if (r.faixa !== faixa) return;\n      const linha = document.createElement("div"); linha.className="spot-rota-item";\n      linha.setAttribute("data-spot-route-item","1"); linha.setAttribute("data-idx",String(idx)); linha.setAttribute("data-name",(r.nome||"").toLowerCase());\n      const cb = document.createElement("input"); cb.type="checkbox"; cb.id="spot-rota-cb-"+idx; cb.className="spot-rota-checkbox"; cb.checked=true;\n      cb.onchange = () => spotApplyFilter(SPOT_QUERY); linha.appendChild(cb);\n      const bolinha = document.createElement("div"); bolinha.className="spot-rota-color"; bolinha.style.backgroundColor=r.cor; linha.appendChild(bolinha);\n      const label = document.createElement("span"); label.className="spot-rota-label";\n      label.textContent = r.nome;\n      label.title = "Clique para centralizar e ver o raio no mapa";\n      label.onclick = () => spotFocusRoute(idx);\n      linha.appendChild(label);\n      const badge = document.createElement("span"); badge.className="spot-raio-badge"; badge.textContent=r.raio||""; linha.appendChild(badge);\n      bloco.appendChild(linha);\n    });\n    container.appendChild(bloco);\n  });\n  const search = document.getElementById("spot-search");\n  if (search) search.addEventListener("input", e => spotApplyFilter(e.target.value));\n  document.getElementById("spot-btn-showall").onclick = spotShowAll;\n  document.getElementById("spot-btn-hideall").onclick = spotHideAll;\n  spotApplyFilter("");\n}\ndocument.addEventListener("DOMContentLoaded", spotInitSidebar);\n</script>\n'
sidebar_html_final = SIDEBAR_TEMPLATE.replace(
    '__SPOT_DATA__',
    json.dumps({'map_var': m.get_name(), 'faixas': labels_ordenados, 'rotas': rota_layers})
)
m.get_root().html.add_child(Element(sidebar_html_final))

map_file = 'mapa_setorizacao_5_1.html'
m.save(map_file)
print('Mapa salvo:', map_file)

from google.colab import files
files.download(map_file)
